In [ ]:
"""
A lightweight interactive form proof of concept using Pydantic and ipywidgets.

This notebook defines a Pydantic model (`UploadMetadata`) for capturing
experiment metadata such as experiment name, investigator, data type,
collection date, and optional S3 bucket target. The model enforces data
validation and type checking.

The code uses `ipywidgets` and `IPython.display` to render simple interactive
form elements in a Jupyter Notebook environment, allowing users to input and
validate metadata directly within the notebook interface.

Usage in VS Code:
1. Install dependencies using `pip install -r "Jupyter Notebook/requirements.txt"`
2. Run all cells in the notebook sequentially or click 'Run All'
3. Enter values in the form fields and click "Submit" to validate
4. Click "Simulate Model Update" to add a new field (`new_testing_field`) to the model
   - After the update, the form refreshes automatically to include the new field
5. Repeat submissions to see validation feedback for the updated model
"""

'\nA lightweight interactive form prototype using Pydantic and ipywidgets.\n\nThis notebook defines a Pydantic model (`UploadMetadata`) for capturing\nexperiment metadata such as experiment name, investigator, data type,\ncollection date, and optional S3 bucket target. The model enforces data\nvalidation and type checking.\n\nThe code uses `ipywidgets` and `IPython.display` to render simple interactive\nform elements in a Jupyter Notebook environment, allowing users to input and\nvalidate metadata directly within the notebook interface.\n'

In [ ]:
from pydantic import BaseModel, Field, ValidationError
from pydantic.fields import PydanticUndefined
from typing import Optional
import ipywidgets as widgets
from IPython.display import display, clear_output

# Initial Pydantic model
class UploadMetadata(BaseModel):
    experiment_name: str = Field(..., title="Experiment Name")
    principal_investigator: str = Field(..., title="Principal Investigator")
    data_type: str = Field(..., title="Data Type (e.g., behavior, ecephys)")
    date_collected: str = Field(..., title="Date Collected (YYYY-MM-DD)")
    s3_bucket_target: Optional[str] = Field("aind-open-data", title="S3 Bucket Target")

CURRENT_MODEL = UploadMetadata


In [3]:
output_box = widgets.Output()
display(output_box)

def build_form(model_cls):
    """
    Build a Jupyter widget form dynamically from a Pydantic model.
    Returns a container widget with input fields and submit/update buttons.
    """
    fields = {}
    for name, field in model_cls.model_fields.items():
        title = field.title or name.replace("_", " ").title()
        if field.default is PydanticUndefined or field.default is None:
            default = ""
        else:
            default = str(field.default)
        fields[name] = widgets.Text(
            value=default,
            description=title,
            layout=widgets.Layout(width='400px')
        )

    submit_btn = widgets.Button(description="Submit", button_style='success')
    update_btn = widgets.Button(description="Simulate Model Update", button_style='info')

    def refresh_form():
        """Clear current form and rebuild with updated model"""
        clear_output(wait=True)
        display(output_box)
        new_form, _ = build_form(CURRENT_MODEL)
        display(new_form)

    def on_submit(btn):
        with output_box:
            clear_output()
            data = {name: w.value for name, w in fields.items()}
            try:
                validated = model_cls(**data)
                print("✅ Success!")
                print(validated.model_dump_json(indent=2))
            except ValidationError as e:
                print("❌ Validation error:")
                print(e.json(indent=2))

    def on_update(btn):
        global CURRENT_MODEL
        if "new_testing_field" not in CURRENT_MODEL.model_fields:
            class UpdatedModel(CURRENT_MODEL):
                new_testing_field: Optional[str] = Field("huzzah!", title="Our New Field!")
            CURRENT_MODEL = UpdatedModel

            with output_box:
                clear_output()
                print("🔄 Model updated — form will refresh automatically!")

            refresh_form()  # rebuild the form with the new field
        else:
            with output_box:
                clear_output()
                print("⚠️ New field already exists — no update performed.")

    submit_btn.on_click(on_submit)
    update_btn.on_click(on_update)

    container = widgets.VBox(list(fields.values()) + [submit_btn, update_btn])
    return container, fields


Output()

In [4]:
form, fields = build_form(CURRENT_MODEL)
display(form)